<a href="https://colab.research.google.com/github/guupiii/ESAA/blob/main/ESAA_OB_week_3_1_review.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**1)) 코드 흐름 요약**

1. 데이터 로드 및 확인

train.info(), train.describe()를 활용하여 데이터 구조 및 변수 타입 확인

결측치 존재 여부 확인

2. EDA 수행

fraud 변수 분포 확인 >> 불균형 데이터 확인

수치형 변수 상관분석을 통해 주요 변수(age_of_driver, annual_income) 간 관계 파악

범주형 변수는 barplot을 통해 fraud와의 관계 분석

카이제곱 검정을 통해 변수 선택

3. EDA 수행

fraud 변수 분포 확인 >> 불균형 데이터 확인

수치형 변수 상관분석을 통해 주요 변수(age_of_driver, annual_income) 간 관계 파악

범주형 변수는 barplot을 통해 fraud와의 관계 분석

카이제곱 검정을 통해 변수 선택

4. 데이터 전처리

결측치가 적은 컬럼은 해당 행 삭제

year 변수는 0,1로 인코딩

이상치(예: age_of_driver = 1) 제거

수치형 변수 >> Standard Scaling 적용

범주형 변수 >> One-Hot Encoding 적용

month, claim_day_of_week >> Cyclic Encoding 적용

SMOTE를 활용한 데이터 불균형 처리

5. 모델 학습

RandomForest, XGBoost, LightGBM 모델 학습

Logistic Regression 모델 추가 학습

6. 모델 평가 및 선택

Accuracy, Precision, Recall, F1-score, ROC-AUC 비교

최종적으로 Logistic Regression 모델 선택

**2)) 새롭게 알게 된 내용 / 어려운 내용 / 배울 점**

1. 새롭게 알게 된 내용

데이터 불균형 문제를 해결하기 위해 SMOTE 기법을 활용할 수 있음을 알게 되었다.

범주형 변수와 타겟 변수 간 관계를 분석할 때 barplot과 카이제곱 검정을 활용할 수 있음을 이해하였다.

2. 어려웠던 내용

어떤 변수를 제거해야 하는지 판단하는 기준을 설정하는 것이 어려웠다.

범주형 변수와 타겟 변수 간 관계를 해석하는 과정이 쉽지 않았다.


In [ ]:
# 수치형 변수 상관분석
num_cols = ['age_of_driver', 'safty_rating', 'annual_income', 'liab_prct', 'policy_report_filed_ind', 'claim_est_payout',
            'age_of_vehicle', 'vehicle_price', 'vehicle_weight']

corr = train[num_cols + ['fraud']].corr(method='pearson')

plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap='coolwarm', center=0)
plt.title('Correlation Matrix', fontsize=14)
plt.show()

In [ ]:
# month 변수에 따른 Fraud 정보 파악하기

month_order = ['JAN', 'FEB', 'MAR', 'APR', 'MAY', 'JUN',
               'JUL', 'AUG', 'SEP', 'OCT', 'NOV', 'DEC']
month_total = train['month'].value_counts().reindex(month_order)
month_fraud = train[train['fraud'] == 1]['month'].value_counts().reindex(month_order)

fig, ax1 = plt.subplots(figsize=(10,6))

sns.barplot(x=month_total.index, y=month_total.values, color='skyblue', ax=ax1)
ax1.set_ylabel('Total Count', color='blue')
ax1.set_xlabel('Month')
ax1.tick_params(axis='y', labelcolor='blue')

ax2 = ax1.twinx()
sns.lineplot(x=month_fraud.index, y=month_fraud.values, marker='o', color='red', ax=ax2)
ax2.set_ylabel('Fraud Count', color='red')
ax2.tick_params(axis='y', labelcolor='red')

plt.title('Monthly Data Count & Fraud Count')
plt.show()

In [ ]:
# SMOTE를 이용한 OverSampling
from imblearn.over_sampling import SMOTE
from sklearn.model_selection import train_test_split

X = train.drop('fraud', axis=1)
y = train['fraud']

X_train, X_valid, y_train, y_valid = train_test_split(X, y, test_size=0.2, random_state=42)

smote = SMOTE(random_state=42)
X_train_res, y_train_res = smote.fit_resample(X_train, y_train)

# 결과 확인
print("SMOTE 전 학습 데이터 클래스 비율:")
print(y_train.value_counts())
print("\nSMOTE 후 학습 데이터 클래스 비율:")
print(y_train_res.value_counts())

print("\n원본 X_train shape:", X_train.shape)
print("SMOTE 후 X_train_res shape:", X_train_res.shape)